# Dataset cleaning (Team Sonoma)

This notebook turns the raw Kaggle files into one cleaned dataset that the whole team loads, so every EDA and model notebook uses the same rows and the same target. We're predicting the grape varietal from the review text.

It loads and merges the raw data, builds the `grape` label, picks which grapes count as classes, hides grape names in the review text, adds a few helper columns, and saves the result.

## 1. Load, merge, and de-duplicate

We use both Kaggle CSVs. The 150k file adds about 50k reviews, but it has no `title` or taster columns, so those are blank for its rows. The `source` column shows which file each review came from.

`source_row` keeps each review's row number in its raw file, so anyone working straight from the Kaggle CSVs (or the 130k JSON, which has the same row order) can join to this file.

Cleaning follows `CSV_download.py`. Duplicates are matched on description + winery, and when a review is in both files we keep the 130k copy since it has more columns. The raw files go in `data/` (gitignored), downloaded from Kaggle's zynicide/wine-reviews.

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = 'data'
CSV_130K = f'{DATA_DIR}/winemag-data-130k-v2.csv'
CSV_150K = f'{DATA_DIR}/winemag-data_first150k.csv'

COLUMNS = ['country', 'description', 'designation', 'points', 'price',
           'province', 'region_1', 'region_2', 'variety', 'winery',
           'title', 'taster_name', 'taster_twitter_handle']
DEDUPE_KEY = ['description', 'winery']

In [2]:
def load_clean(path, source):
    """Read one raw Kaggle file and clean it like CSV_download.py."""
    df = pd.read_csv(path, index_col=0).reindex(columns=COLUMNS)
    df['source'] = source
    df['source_row'] = df.index  # row number in the raw file, for joining to raw-file work
    n_raw = len(df)

    text_cols = df.select_dtypes(include=['object', 'str']).columns
    df[text_cols] = df[text_cols].apply(lambda s: s.str.strip()).replace('', np.nan)

    df = df.dropna(subset=['description', 'points'])
    df = df[df['points'].between(80, 100)].copy()
    df.loc[df['price'] <= 0, 'price'] = np.nan

    df = df.drop_duplicates().drop_duplicates(subset=DEDUPE_KEY)
    print(f'{source}: {n_raw:,} raw rows -> {len(df):,} after cleaning and de-duplication')
    return df


df_130k = load_clean(CSV_130K, '130k')
df_150k = load_clean(CSV_150K, '150k')

130k: 129,971 raw rows -> 119,976 after cleaning and de-duplication


150k: 150,930 raw rows -> 97,839 after cleaning and de-duplication


In [3]:
# 130k rows go first so they win cross-file duplicates (they have title and taster).
reviews = (
    pd.concat([df_130k, df_150k], ignore_index=True)
    .drop_duplicates(subset=DEDUPE_KEY, keep='first')
    .reset_index(drop=True)
)

print(f'Merged: {len(reviews):,} unique reviews')
print(reviews['source'].value_counts().to_string())

Merged: 169,613 unique reviews
source
130k    119976
150k     49637


In [4]:
# What's missing, and how much of it comes from the 150k-only rows.
missing = pd.DataFrame({
    'missing_all': reviews.isna().sum(),
    'missing_130k': reviews[reviews['source'].eq('130k')].isna().sum(),
    'missing_150k': reviews[reviews['source'].eq('150k')].isna().sum(),
})
missing['pct_all'] = (missing['missing_all'] / len(reviews) * 100).round(1)
missing

,missing_all,missing_130k,missing_150k,pct_all
country,60,59,1,0.0
description,0,0,0,0.0
designation,50119,34540,15579,29.5
points,0,0,0,0.0
price,12816,8395,4421,7.6
province,60,59,1,0.0
region_1,27959,19559,8400,16.5
region_2,102761,73213,29548,60.6
variety,1,1,0,0.0
winery,0,0,0,0.0


## 2. Build the target

Each raw `variety` is looked up in `variety_map.csv`, which merges synonyms into one `grape` name (Shiraz → Syrah, Pinot Grigio → Pinot Gris, Primitivo → Zinfandel). It also adds `color`, `category` (still, sparkling, dessert), `is_blend`, and a `label_type` saying what kind of label it is. The map is a plain CSV so anyone on the team can review or edit it.

Blends get no `grape`, since we're modeling single grapes. Portuguese Red and Portuguese White do count as classes, which the assignment allows. Styles like rosé, sparkling and Port get a color and category but no grape. Ambiguous labels such as "Cabernet" and "Tokay" also get no grape. Rare labels nobody has checked are marked `reviewed = False`; none of them is big enough to become a class.

We also keep the team's `style_group` and `batch` from `wine_style_groups_final.csv`, so the batch EDA can filter this file the same way.

In [5]:
variety_map = pd.read_csv('variety_map.csv').drop(columns='n_reviews')

unmapped = set(reviews['variety'].dropna()) - set(variety_map['variety'])
assert not unmapped, f'Add these to variety_map.csv: {sorted(unmapped)}'

reviews = reviews.merge(variety_map, on='variety', how='left')
# The one review with no variety has no map row; treat it as unlabeled.
reviews[['is_blend', 'reviewed']] = reviews[['is_blend', 'reviewed']].fillna(False).astype(bool)
print(f'{reviews["grape"].notna().sum():,} of {len(reviews):,} reviews have a grape label')
reviews.groupby('label_type', dropna=False).size().rename('reviews').to_frame()

126,554 of 169,613 reviews have a grape label


,reviews
label_type,
ambiguous,29
blend,33143
grape,121177
regional,4216
style,9886
unreviewed,1161
NaN,1


In [6]:
# The team's style groups, kept so batch EDA lines up with this file.
style_map = (
    pd.read_csv('wine_style_groups_final.csv')
    .assign(variety=lambda d: d['varieties'].str.split(';'))
    .explode('variety')
    .assign(variety=lambda d: d['variety'].str.strip())
    [['variety', 'style_group', 'batch']]
)
assert not style_map['variety'].duplicated().any(), 'a variety is listed in two style groups'

reviews = reviews.merge(style_map, on='variety', how='left')
reviews['batch'] = reviews['batch'].astype('Int64')
reviews.groupby(['batch', 'style_group']).size().rename('reviews').to_frame()

reviews
batch style_group                                       
1     Greek & Other Whites                           380
      Italian Whites                                2740
      Malbec                                        3446
      Other Reds                                     767
      Pinot Noir                                   16943
      Rhône & Grenache Reds                         5579
      Rosé                                          4109
2     Aromatic Whites                               3008
      Chardonnay                                   15647
      Fortified & Dessert                           1402
      Sangiovese                                    4799
      Sparkling                                     5771
      White Blend (generic)                         3224
3     Austrian & German Whites                      2109
      Cabernet Sauvignon                           13290
      Chenin, Semillon & Loire/SW France Whites     3325
      Merlot                                        4757
      Nebbiolo                                      3126
      Sauvignon Blanc                               7290
4     Bordeaux-style Red Blend                     13243
      Iberian Reds                                  4506
      Pinot Gris & Pinot Blanc                      4028
      Rhône Whites                                  2584
      Riesling                                      6663
      Tempranillo                                   2960
5     Iberian Whites                                3213
      Italian Reds                                  4875
      Light & Central European Reds                 1975
      Red Blend (generic)                          12206
      Syrah                                         7548
      Zinfandel                                     4099

In [7]:
# Reviews per canonical grape (before any minimum-size cut).
grape_counts = (
    reviews.dropna(subset=['grape'])
    .groupby(['grape', 'color'], dropna=False)
    .size()
    .rename('reviews')
    .sort_values(ascending=False)
    .reset_index()
)
grape_counts.head(40)

,grape,color,reviews
0,Pinot Noir,red,16942
1,Chardonnay,white,15647
2,Cabernet Sauvignon,red,13274
3,Syrah,red,7548
4,Sauvignon Blanc,white,7290
5,Riesling,white,6662
6,Sangiovese,red,4799
7,Merlot,red,4757
8,Zinfandel,red,4099
9,Malbec,red,3446


## 3. Choose the classes

A grape needs at least 500 reviews to be a class, which leaves 29 classes. Everything else stays in the file with `in_target = False`, so models filter on that column and EDA can still use all the reviews. The regional models (e.g. Bordeaux Merlot vs Cabernet) are smaller, so check class sizes again after filtering.

In [8]:
MIN_CLASS_SIZE = 500

class_sizes = reviews['grape'].value_counts()
target_grapes = class_sizes[class_sizes >= MIN_CLASS_SIZE].index.tolist()
reviews['in_target'] = reviews['grape'].isin(target_grapes)

print(f'{len(target_grapes)} classes with at least {MIN_CLASS_SIZE} reviews')
print(f'{reviews["in_target"].sum():,} of {len(reviews):,} reviews are in a target class')

(class_sizes[target_grapes]
 .rename('reviews')
 .to_frame()
 .join(reviews.drop_duplicates('grape').set_index('grape')[['color', 'label_type']]))

29 classes with at least 500 reviews
113,379 of 169,613 reviews are in a target class


,reviews,color,label_type
grape,,,
Pinot Noir,16942,red,grape
Chardonnay,15647,white,grape
Cabernet Sauvignon,13274,red,grape
Syrah,7548,red,grape
Sauvignon Blanc,7290,white,grape
Riesling,6662,white,grape
Sangiovese,4799,red,grape
Merlot,4757,red,grape
Zinfandel,4099,red,grape


In [9]:
# Just under the cut, so the team can see what was left out.
class_sizes[class_sizes < MIN_CLASS_SIZE].head(10).rename('reviews').to_frame()

,reviews
grape,
Nero d'Avola,478
Aglianico,459
Mourvèdre,448
Verdejo,368
Garganega,361
Petit Verdot,359
Torrontés,348
Blaufränkisch,343
Melon,281


## 4. Mask grape names in the review text

20–40% of reviews name their own grape ("This Merlot...", "a juicy Zin"), which would let a classifier cheat. `description_masked` replaces every grape name, synonym and common nickname with `varietyname`. The original `description` is kept so we can see how much accuracy depends on the names.

"Melon" is left alone because it's also a tasting note. Place names like Barolo, Chianti and Rioja are left in on purpose; how much they help the model is something we want to measure.

In [10]:
import re
import unicodedata

NICKNAMES = ['Cab', 'Cab Franc', 'Cab Sauv', 'Zin', 'Chard', 'Sauv', 'Sauv Blanc',
             'Gewürz', 'Grüner', 'Veltliner', 'Pinot', 'Cabernet', 'Chenin', 'Petite']
NOT_MASKED = {'Melon'}


def strip_accents(text):
    return ''.join(ch for ch in unicodedata.normalize('NFKD', text)
                   if not unicodedata.combining(ch))


checked = variety_map[variety_map['label_type'].isin(['grape', 'ambiguous'])]
names = set(checked['variety']) | set(checked['grape'].dropna()) | set(NICKNAMES)
names = {n for n in names if n not in NOT_MASKED}
names |= {strip_accents(n) for n in names}

# Longest names first, so 'Cabernet Sauvignon' is replaced as one unit before 'Cabernet'.
mask_terms = sorted(names, key=len, reverse=True)
MASK_PATTERN = re.compile(
    r'\b(?:' + '|'.join(re.escape(t) for t in mask_terms) + r')s?\b', flags=re.IGNORECASE
)
MASK_TOKEN = 'varietyname'

reviews['description_masked'] = reviews['description'].str.replace(MASK_PATTERN, MASK_TOKEN, regex=True)

changed = reviews['description'].ne(reviews['description_masked'])
print(f'{len(mask_terms):,} names masked; {changed.mean():.1%} of reviews changed')

252 names masked; 37.4% of reviews changed


In [11]:
# Check: how often each target grape's own name survives masking (should be ~0%).
def own_name_rate(text_col):
    rates = {}
    for grape in target_grapes:
        if grape.startswith('Portuguese'):
            continue
        text = reviews.loc[reviews['grape'].eq(grape), text_col]
        pattern = r'\b' + re.escape(strip_accents(grape).split()[0])
        rates[grape] = text.map(strip_accents).str.contains(pattern, case=False, regex=True).mean()
    return pd.Series(rates)

(pd.DataFrame({'before': own_name_rate('description'), 'after': own_name_rate('description_masked')})
 .mul(100).round(1).sort_values('before', ascending=False))

,before,after
Barbera,46.7,0.0
Merlot,43.3,0.0
Riesling,42.8,0.0
Viognier,41.6,0.0
Petite Sirah,39.5,0.0
Malbec,35.8,0.0
Pinot Noir,35.4,0.1
Syrah,33.6,0.0
Pinot Blanc,33.1,0.0
Pinot Gris,33.0,0.0


In [12]:
# A few examples, before and after.
with pd.option_context('display.max_colwidth', None):
    display(reviews.loc[changed, ['grape', 'description', 'description_masked']].sample(5, random_state=0))

,grape,description,description_masked
57086,Merlot,"This isn't a wine to age. It's soft and appealing in its charms, offering drink-me-now cherries, red currants, herbs and sandalwood. The winemaker blended it with a little Cabernet Sauvignon and Petite Verdot, which gives it structural and flavor complexities.","This isn't a wine to age. It's soft and appealing in its charms, offering drink-me-now cherries, red currants, herbs and sandalwood. The winemaker blended it with a little varietyname and varietyname, which gives it structural and flavor complexities."
128344,NaN,"Benuara is a 70-30 blend of Nero d'Avola and Syrah with a strongly menthol-driven nose enhanced by dark fruit, spice and green olive. Pair this wine with tapenade-crusted chicken.","Benuara is a 70-30 blend of varietyname and varietyname with a strongly menthol-driven nose enhanced by dark fruit, spice and green olive. Pair this wine with tapenade-crusted chicken."
30740,Syrah,"There's a spicy soul to this beautiful Syrah, as forward notes of white pepper, licorice and sweet spice abound on the nose and mouth, infusing the raspberry, cherry and brambly blackberry fruits throughout. Additional notes of purple violets, fynbos and black tea add depth and interest. Medium weight, the approachable palate boasts a crushed satin texture, with fine yet gripping tannins that hang on through the spicy finish. Drink now–2020.","There's a spicy soul to this beautiful varietyname, as forward notes of white pepper, licorice and sweet spice abound on the nose and mouth, infusing the raspberry, cherry and brambly blackberry fruits throughout. Additional notes of purple violets, fynbos and black tea add depth and interest. Medium weight, the approachable palate boasts a crushed satin texture, with fine yet gripping tannins that hang on through the spicy finish. Drink now–2020."
52366,NaN,"Pure Touriga Nacional, this is a crisp and fruity wine. Its red currants and light tannic structure give it shape and bring out the fruitiness that makes it immediately attractive. Ready to drink now.","Pure varietyname, this is a crisp and fruity wine. Its red currants and light tannic structure give it shape and bring out the fruitiness that makes it immediately attractive. Ready to drink now."
38444,Chardonnay,"On the nose, this aging Chardonnay brings corn to mind. Its palate is neither exciting nor flawed, with melon and stone fruit flavors that are yeasty and short, finishing on a bready and slightly oxidized note. Not a great value for its price.","On the nose, this aging varietyname brings corn to mind. Its palate is neither exciting nor flawed, with melon and stone fruit flavors that are yeasty and short, finishing on a bready and slightly oxidized note. Not a great value for its price."


## 5. Extra columns

`review_id` numbers rows the same way `CSV_download.py` does, so it should match the team database. `word_count` is words per review. `split` is a fixed 70/15/15 train/val/test assignment, stratified by grape (seed 4401), so every model is tested on the same held-out reviews. We don't parse vintage; we don't need it, and it's Team Napa's target.

In [13]:
SEED = 4401
SPLIT_FRACTIONS = {'train': 0.70, 'val': 0.15, 'test': 0.15}

reviews.insert(0, 'review_id', range(1, len(reviews) + 1))
reviews['word_count'] = reviews['description'].str.split().str.len()

rng = np.random.default_rng(SEED)
reviews['split'] = ''
for _, idx in reviews.groupby(reviews['grape'].fillna('(none)')).groups.items():
    idx = rng.permutation(idx)
    n_train = round(len(idx) * SPLIT_FRACTIONS['train'])
    n_val = round(len(idx) * SPLIT_FRACTIONS['val'])
    reviews.loc[idx[:n_train], 'split'] = 'train'
    reviews.loc[idx[n_train:n_train + n_val], 'split'] = 'val'
    reviews.loc[idx[n_train + n_val:], 'split'] = 'test'

pd.crosstab(reviews['in_target'], reviews['split'], margins=True)

split,test,train,val,All
in_target,,,,
False,8454,39381,8399,56234
True,17010,79363,17006,113379
All,25464,118744,25405,169613


## 6. Save

The result goes to `data/wine_clean.parquet`, which is gitignored like the raw data. Run this notebook to rebuild it, then load it with:

```python
wines = pd.read_parquet('data/wine_clean.parquet')
model_data = wines[wines['in_target']]
```

In [14]:
OUTPUT_PATH = f'{DATA_DIR}/wine_clean.parquet'

column_order = [
    'review_id', 'grape', 'in_target', 'split', 'label_type', 'is_blend', 'color', 'category',
    'style_group', 'batch',
    'description', 'description_masked', 'word_count', 'variety',
    'country', 'province', 'region_1', 'region_2', 'winery', 'designation', 'title',
    'points', 'price', 'taster_name', 'taster_twitter_handle', 'source', 'source_row', 'reviewed',
]
assert set(column_order) == set(reviews.columns), set(reviews.columns) ^ set(column_order)

reviews[column_order].to_parquet(OUTPUT_PATH, index=False)
print(f'Saved {len(reviews):,} reviews x {len(column_order)} columns to {OUTPUT_PATH}')

Saved 169,613 reviews x 28 columns to data/wine_clean.parquet
